In [16]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
import numpy as np
import os
from glob import glob
from mpl_toolkits.basemap import Basemap
from numpy import meshgrid
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib import colormaps
import pandas as pd
import math
from datetime import datetime
import datetime as dt
from ridgeplot import ridgeplot
import joypy
import seaborn as sns
from matplotlib import cm
import climpred
from xclim import sdba
from climpred.options import OPTIONS
import json
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from matplotlib.lines import Line2D  # For custom legend entries
import warnings
from sklearn.exceptions import UndefinedMetricWarning
import matplotlib.gridspec as gridspec
import hydroeval as he
import re
from matplotlib.colors import Normalize
import matplotlib.colors as mcolors

from function import preprocessUtils as putils
from function import masks
from function import verifications
from function import funs as f
from function import conf
from function import loadbias
from function import quikplot as qp
from function import dataLoad
from function import conf


warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

global dim_order, region_name, test_year, leads_
dim_order = conf.dim_order

test_year = 2019
leads_ = [6,13,20,27]

dir = '/glade/work/klesinger/FD_RZSM_deep_learning'
assert test_year == 2019, 'This is only the script for when the testing years are 2018-2019. Test year must = 2019.'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## This script only works if there is a single experiment done for china and australia
### We are doing EX29 only
### For both ECMWF and GEFSv12
## For both ERA5-land and GLEAM

In [17]:
region_name = 'australia' #['australia','china','CONUS']
obs_source = 'ERA5' #['GLEAM','ERA5']

if obs_source == 'ERA5':
    soil_dir = conf.era_data
elif obs_source == 'GLEAM':
    soil_dir = conf.gleam_data

In [18]:
mask, mask_anom = masks.load_mask_vals(region_name)
try:
    mask = mask.rename({'X':'longitude','Y':'latitude'})
except ValueError:
    pass

In [19]:
global custom_names
'''This is for the final plot for ACC and CRPS'''
custom_names = {
    'GEFSv12': 'GEFSv12','GEFSv12-BC': 'GEFSv12-BC', 'DL-DM_GEFSv12': 'DL-DM-GEFSv12','DL-DM_ECMWF': 'DL-DM-ECMWF',
    'ECMWF':'ECMWF', 'ECMWF-BC':'ECMWF-BC',
}

        
def return_name(name):
    if 'XGBOOST' in name:
        name_out = 'ML_NWP_OBS'
    else:
        name_out = name
    custom_names = {name: name_out}

    return(custom_names)

In [20]:
global obs_anomaly,obs_raw
obs_anomaly,obs_raw = dataLoad.load_rzsm_observations(soil_dir, region_name)
obs_anomaly["time"] = obs_anomaly["time"].dt.floor("D")
obs_raw["time"] = obs_raw["time"].dt.floor("D")

try:
    obs_anomaly = obs_anomaly.rename({'X':'longitude','Y':'latitude'})
except ValueError:
    pass

In [21]:
gef_BC, ecm_BC = loadbias.load_additive_bias_anomaly(leads=leads_,region_name=region_name,obs_source=obs_source)

In [22]:

def simulate_metric_bootstrap_all_seasons(metric_,forecast, num_iterations, name_of_forecast,region_name, obs_source):
    
    save_dir = f'Data/{metric_}_bootstrap/{region_name}'
    os.makedirs(save_dir, exist_ok=True)
    save_file = f'{save_dir}/all_season_skill_{name_of_forecast}_forecast_{obs_source}_obs.nc'

    if os.path.exists(save_file):
        print(f'Simulating {num_iterations} bootstraps for {metric_} for forecast {name_of_forecast}')
        obs = verifications.rename_obs_for_climpred(obs_anomaly)
        forecast['lead'].attrs['units'] = 'days'
        skill = verifications.create_climpred_CRPSS_bootstrap(forecast, obs, num_iterations, metric_)
        skill.to_netcdf(save_file)
    else:
        print(f'Already completed {num_iterations} bootstraps for {metric_} for forecast {name_of_forecast}')



def simulate_metric_bootstrap_split_seasons(metric_, forecast, num_iterations, name_of_forecast, region_name, obs_source):
    seasons = f.return_seasons()

    for season_name, season_months in seasons.items():
        save_dir = f'Data/{metric_}_bootstrap/{region_name}'
        os.makedirs(save_dir, exist_ok=True)
        save_file = f'{save_dir}/{season_name}_skill_{name_of_forecast}_forecast_{obs_source}_obs.nc'

        if os.path.exists(save_file):
            print(f'[✔] Already completed {num_iterations} bootstraps for {season_name} {metric_} for forecast {name_of_forecast}')
            continue

        print(f'[⏳] Simulating {num_iterations} bootstraps for {season_name} {metric_} for forecast {name_of_forecast}')
        
        # Subset forecast to season months based on initialization time
        if 'init' in forecast.dims or 'init' in forecast.coords:
            init_months = xr.DataArray(forecast['init'].dt.month, coords={'init': forecast['init']})
            season_forecast = forecast.sel(init=init_months.isin(season_months))
        else:
            raise ValueError("Forecast must have a time dimension 'init' to filter by season.")

        # Prepare obs data (assumes obs_anomaly is defined globally or in scope)
        obs = verifications.rename_obs_for_climpred(obs_anomaly)
        season_forecast['lead'].attrs['units'] = 'days'

        skill = verifications.create_climpred_CRPSS_bootstrap(season_forecast, obs, num_iterations, metric_)
        skill.to_netcdf(save_file)
        print(f'[💾] Saved skill for {season_name} to {save_file}')


In [23]:
# #First run simulation of the bias corrected results (ALL SEASONS)
simulate_metric_bootstrap_all_seasons(metric_='crpss',forecast=gef_BC, num_iterations=1000, name_of_forecast='GEFSv12-BC',region_name=region_name, obs_source=obs_source)

simulate_metric_bootstrap_all_seasons(metric_='crpss',forecast=ecm_BC, num_iterations=1000, name_of_forecast='ECMWF-BC',region_name=region_name, obs_source=obs_source)

#SPLIT SEASONS
simulate_metric_bootstrap_split_seasons(metric_='crpss',forecast=gef_BC, num_iterations=1000, name_of_forecast='GEFSv12-BC',region_name=region_name, obs_source=obs_source)

simulate_metric_bootstrap_split_seasons(metric_='crpss',forecast=ecm_BC, num_iterations=1000, name_of_forecast='ECMWF-BC',region_name=region_name, obs_source=obs_source)


In [24]:
def common_UNET_experiments(correct_experiments, obs_source):
    only_RZSM = [j for j in correct_experiments if 'RZSM' in j] 
    only_ensemble= [j for j in only_RZSM if 'final' not in j]
    only_ensemble = [j for j in only_ensemble if 'Residual' not in j]
    only_ensemble = [j for j in only_ensemble if 'mse' not in j]
    only_2019 = [j for j in only_ensemble if '2012' not in j]
    if obs_source == 'GLEAM':
        only_2019 = [j for j in only_ensemble if 'ERA5' not in j]
    elif obs_source == 'ERA5':
        only_2019 = [j for j in only_ensemble if 'ERA5' in j]
    return(only_2019)

def common_UNET_no_regular_experiments(correct_experiments):
    only_RZSM = [j for j in correct_experiments if 'RZSM' in j] 
    only_ensemble= [j for j in only_RZSM if 'final' not in j]
    only_ensemble = [j for j in only_ensemble if 'Residual' not in j]
    only_ensemble = [j for j in only_ensemble if 'regular' not in j]
    only_2019 = [j for j in only_ensemble if '2012' not in j]
    return(only_2019)

In [25]:


def filter_files_by_ex_GEFS(file_list, color_list, week_,obs_source):
    filtered_files = []
    
    for file in file_list:
        if obs_source== 'GLEAM':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_regular_RZSM', file)
        elif obs_source == 'ERA5':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_regular_ERA5_RZSM', file)
        if match:
            ex_value = f"EX{match.group(1)}"
            if ex_value in color_list:
                filtered_files.append(file)
    
    return filtered_files

def filter_files_by_ex_ECMWF(file_list, color_list, week_,obs_source):
    filtered_files = []
    
    for file in file_list:
        if obs_source =='GLEAM':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_ECMWF_regular_RZSM', file)
        elif obs_source == 'ERA5':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_ECMWF_regular_ERA5_RZSM', file)
        if match:
            ex_value = f"EX{match.group(1)}"
            if ex_value in color_list:
                filtered_files.append(file)
    
    return filtered_files

def return_file_list_by_category(region_name, week_,obs_source):
    black = ['EX0','EX13']
    red = ['EX14','EX15','EX16','EX17','EX22','EX23','EX24','EX25']
    blue = ['EX1','EX2','EX3','EX4','EX5','EX6','EX7','EX8','EX9','EX10','EX11','EX12',
           'EX18','EX19','EX20','EX21','EX27','EX28','EX29']
    
    unet_files = sorted(glob(f'predictions/{region_name}/Wk{week_}_testing/*')) #With a specific subset of data
    #First find the correct experiments
    bias_correction_black_G = filter_files_by_ex_GEFS(unet_files, black, week_,obs_source)
    hybrid_blue_G = filter_files_by_ex_GEFS(unet_files, blue, week_,obs_source)
    obs_red_G = filter_files_by_ex_GEFS(unet_files, red, week_,obs_source)

    bias_correction_black_E = filter_files_by_ex_ECMWF(unet_files, black, week_,obs_source)
    hybrid_blue_E = filter_files_by_ex_ECMWF(unet_files, blue, week_,obs_source)
    obs_red_E = filter_files_by_ex_ECMWF(unet_files, red, week_,obs_source)
    
    return bias_correction_black_G, hybrid_blue_G, obs_red_G, bias_correction_black_E, hybrid_blue_E, obs_red_E


def create_empty_array(ecmwf_acc, day_num):
    u_acc = ecmwf_acc.sel(lead=day_num).copy(deep=True)
    u_crps = ecmwf_acc.sel(lead=day_num).copy(deep=True)
    u_acc[putils.xarray_varname(u_acc)][:,:] = 0
    u_crps[putils.xarray_varname(u_crps)][:,:] = 0
    return u_acc, u_crps

In [26]:
def load_UNET_all_weeks(gefs, files, region_name, new_source,test_year,experiment_number,mask_anom):
    print("Loading UNET testing predictions and reversing the min-max scaling to be back to anomalies")
    add_to_file = gefs.copy(deep=True)

    print(f'Working on files {files} for UNET')
    
    for idx,file in enumerate(files):
        day_num = ((idx+1) * 7) - 1  # Lead time in days
        print(f'Day number is {day_num}')
    
        load_ = np.load(file)[-1,:,:,:,0] 
        test = verifications.reverse_min_max_scaling(load_, region_name, day_num,new_source,test_year)
        test = np.reshape(test,(test.shape[0]//11,11,test.shape[1],test.shape[2]))
        test = np.expand_dims(test, -1)
        
        load_ =  np.reshape(test,(test.shape[0], test.shape[1], test.shape[-1], test.shape[2], test.shape[3]))
        # print(load_.shape)
        load_ = np.where(np.isnan(mask_anom),np.nan,load_)
        
        add_to_file[putils.xarray_varname(add_to_file)][:,:,idx,:,:] = load_[:,:,0,:,:]

    return(add_to_file)



In [27]:
def run_UNET_bootstrap(region_name, obs_anomaly, gef_BC, mask_anom,experiment_number='EX29'):

    #test
    #experiment_number = 'EX29'

    ex29_gefs = []
    ex29_ecmwf = []
    for idx,week in enumerate([1,2,3,4]):
        #We only need a single week for this function to retrieve all data
        bias_correction_black_G, hybrid_blue_G, obs_red_G, bias_correction_black_E, hybrid_blue_E, obs_red_E  = return_file_list_by_category(region_name=region_name, week_=week,obs_source=obs_source)
        ex29_gefs.append([i for i in hybrid_blue_G if experiment_number in i][0])
        ex29_ecmwf.append([i for i in hybrid_blue_E if experiment_number in i][0])
        
    #Now we have all the weeks we need
    # create a merged forecast file
    for new_source in ['ECMWF', 'GEFSv12']:
        if new_source == 'ECMWF':
            files = ex29_ecmwf
        else:
            files = ex29_gefs

            
        forecast = load_UNET_all_weeks(
            gefs=gef_BC, files=files, region_name=region_name,
            new_source='GEFSv12', test_year=test_year,
            experiment_number=experiment_number,mask_anom = mask_anom)
        #First run simulation of the bias corrected results

        simulate_metric_bootstrap_all_seasons(metric_='crpss',forecast=forecast, num_iterations=1000, name_of_forecast=f'{experiment_number}_{new_source}',region_name=region_name, obs_source=obs_source)
        simulate_metric_bootstrap_split_seasons(metric_='crpss',forecast=forecast, num_iterations=1000, name_of_forecast=f'{experiment_number}_{new_source}',region_name=region_name, obs_source=obs_source)
        
    return forecast
                
run_UNET_bootstrap(region_name, obs_anomaly, gef_BC, mask_anom,experiment_number='EX29')

Loading UNET testing predictions and reversing the min-max scaling to be back to anomalies
Working on files ['predictions/australia/Wk1_testing/Wk1_testing_EX29_ECMWF_regular_ERA5_RZSM.npy', 'predictions/australia/Wk2_testing/Wk2_testing_EX29_ECMWF_regular_ERA5_RZSM.npy', 'predictions/australia/Wk3_testing/Wk3_testing_EX29_ECMWF_regular_ERA5_RZSM.npy', 'predictions/australia/Wk4_testing/Wk4_testing_EX29_ECMWF_regular_ERA5_RZSM.npy'] for UNET
Day number is 6
Day number is 13
Day number is 20
Day number is 27
Simulating 1000 bootstraps for crpss for forecast EX29_ECMWF
Forecast dataset dims: Frozen({'lon': 96, 'lat': 48, 'lead': 4, 'member': 11, 'init': 104})
Observations dataset dims: Frozen({'time': 7731, 'lat': 48, 'lon': 96})


/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})


[⏳] Simulating 1000 bootstraps for DJF crpss for forecast EX29_ECMWF
Forecast dataset dims: Frozen({'lon': 96, 'lat': 48, 'lead': 4, 'member': 11, 'init': 26})
Observations dataset dims: Frozen({'time': 7731, 'lat': 48, 'lon': 96})


/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})


[💾] Saved skill for DJF to Data/crpss_bootstrap/australia/DJF_skill_EX29_ECMWF_forecast_ERA5_obs.nc
[⏳] Simulating 1000 bootstraps for MAM crpss for forecast EX29_ECMWF
Forecast dataset dims: Frozen({'lon': 96, 'lat': 48, 'lead': 4, 'member': 11, 'init': 26})
Observations dataset dims: Frozen({'time': 7731, 'lat': 48, 'lon': 96})


...

/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})


[💾] Saved skill for JJA to Data/crpss_bootstrap/australia/JJA_skill_EX29_GEFSv12_forecast_ERA5_obs.nc
[⏳] Simulating 1000 bootstraps for SON crpss for forecast EX29_GEFSv12
Forecast dataset dims: Frozen({'lon': 96, 'lat': 48, 'lead': 4, 'member': 11, 'init': 26})
Observations dataset dims: Frozen({'time': 7731, 'lat': 48, 'lon': 96})


/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})


[💾] Saved skill for SON to Data/crpss_bootstrap/australia/SON_skill_EX29_GEFSv12_forecast_ERA5_obs.nc


<xarray.Dataset>
Dimensions:  (lon: 96, lat: 48, lead: 4, member: 11, init: 104)
Coordinates:
  * lon      (lon) float64 112.0 112.5 113.0 113.5 ... 158.0 158.5 159.0 159.5
  * lat      (lat) float64 -13.0 -13.5 -14.0 -14.5 ... -35.0 -35.5 -36.0 -36.5
  * lead     (lead) int64 6 13 20 27
  * member   (member) int64 0 1 2 3 4 5 6 7 8 9 10
  * init     (init) datetime64[ns] 2018-01-03 2018-01-10 ... 2019-12-25
Data variables:
    RZSM     (init, member, lead, lat, lon) float32 ...
Attributes:
    lead:     days

In [ ]:

stop